In [ ]:
root_path = '/gpfs/home/pl2948/P_KNN/revision'

In [ ]:
import pandas as pd
import os

df = pd.read_csv(os.path.join(root_path, "data/output/variants_output.tsv"), sep="\t", low_memory=False)

print(df.shape)
print(df.columns.tolist())

In [ ]:
# 跳過 ## 開頭的 header 行
func_df = pd.read_csv(
    os.path.join(root_path, "data/output/ENIGMA_BRCA12_functional_assays_scores.sorted.hg38.vcf"),
    sep="\t",
    comment="#",
    header=None,
    names=["CHROM", "POS", "ID", "REF", "ALT", "QUAL", "FILTER", "INFO"]
)

print(func_df.shape)
func_df.head(3)

In [ ]:
func_df.INFO[0]

In [ ]:
# 先看 variant_type 有哪些值
print(df["variant_type"].value_counts().head(20))
print()
# 也看看 result 欄位實際上長什麼樣
result_cols = [c for c in df.columns if c.startswith("result_") and "enigma" in c]
print("Result cols:", result_cols)
print()
# 看看這些欄位有多少非空值
print(df[result_cols].notna().sum())
print()
# 看看非空的值長什麼樣子
for col in result_cols:
    vals = df[col].dropna().unique()
    if len(vals) > 0:
        print(f"{col}: {vals[:5]}")

In [ ]:
# ── 1. 各 assay 的 LOF / Functional mapping ──────────────────────────
assay_map = {
    # ── BRCA1 assays ───────────────────────────────────────────────────
    "result_findlay_enigma_brca12_functional_assays": {
        "LOF":          ["LOF"],
        "Functional":   ["FUNC"],
        "Intermediate": ["INT"],
    },
    "result_starita_enigma_brca12_functional_assays": {
        # HR score: 0.0 = non-functional, 4.0 = functional, 1.0 = intermediate
        "LOF":          ["0.0"],
        "Functional":   ["4.0"],
        "Intermediate": ["1.0"],
    },
    "result_petitalot_enigma_brca12_functional_assays": {
        # 1P = functional, 2P = intermediate, 3P = LOF
        "LOF":          ["3P"],
        "Functional":   ["1P"],
        "Intermediate": ["2P"],
    },
    "result_fernandes_enigma_brca12_functional_assays": {
        # IARC class: 1.0/2.0 = benign, 4.0/5.0 = pathogenic, 3.0 = VUS
        "LOF":          ["4.0", "5.0"],
        "Functional":   ["1.0", "2.0"],
        "Intermediate": ["3.0"],
    },
    "result_bouwman1_enigma_brca12_functional_assays": {
        "LOF":          ["Deleterious"],
        "Functional":   ["Neutral", "Likely not pathogenic"],
        "Intermediate": ["Not Clear"],
    },
    # ── BRCA2 assays ───────────────────────────────────────────────────
    "result_mesman_enigma_brca12_functional_assays": {
        "LOF":          ["Loss of function"],
        "Functional":   ["Functional"],
        "Intermediate": ["Intermediate function"],
    },
    "result_richardson_enigma_brca12_functional_assays": {
        "LOF":          ["Damaging"],
        "Functional":   ["Neutral"],
    },
    "result_biwas_enigma_brca12_functional_assays": {
        "LOF":          ["(NF)"],
        "Functional":   ["(F)"],
        "Intermediate": ["(I)"],
    },
    "result_ikegami_enigma_brca12_functional_assays": {
        "LOF":          ["fClass 4 or 5"],
        "Functional":   ["fClass 1", "fClass 1 or 2"],
        "Intermediate": ["fClass 3"],
    },
    # result_bouwman2: 只有 "Many Provided"，無法 binary mapping → 略過
}

# ── 2. 統一 label 函數 ────────────────────────────────────────────────
from collections import Counter

def parse_functional_label(row):
    calls = []
    for col, mapping in assay_map.items():
        val = str(row[col]).strip()
        if val == "-" or val == "" or val == "nan":
            continue
        for label, keywords in mapping.items():
            if val in keywords:
                calls.append(label)
                break
    if not calls:
        return None
    counts = Counter(calls)
    top, top_n = counts.most_common(1)[0]
    # 若有衝突（LOF 和 Functional 都出現），標記 Conflicting
    if "LOF" in counts and "Functional" in counts:
        return "Conflicting"
    return top

# ── 3. 篩選 substitution（= missense 的 variant_type）─────────────────
is_sub = df["variant_type"] == "substitution"

# 確認是 missense 用 varconsequences 或 protein 欄位
# 先看看有沒有 varconsequences
print(df["varconsequences"].value_counts().head(10))

In [ ]:
from collections import Counter

# ── 1. Assay mapping ──────────────────────────────────────────────────
assay_map = {
    # ── BRCA1 assays ───────────────────────────────────────────────────
    "result_findlay_enigma_brca12_functional_assays": {
        "LOF":          ["LOF"],
        "Functional":   ["FUNC"],
        "Intermediate": ["INT"],
    },
    "result_starita_enigma_brca12_functional_assays": {
        # HR score: 0.0 = non-functional, 4.0 = functional, 1.0 = intermediate
        "LOF":          ["0.0"],
        "Functional":   ["4.0"],
        "Intermediate": ["1.0"],
    },
    "result_petitalot_enigma_brca12_functional_assays": {
        # 1P = functional, 2P = intermediate, 3P = LOF
        "LOF":          ["3P"],
        "Functional":   ["1P"],
        "Intermediate": ["2P"],
    },
    "result_fernandes_enigma_brca12_functional_assays": {
        # IARC class: 1.0/2.0 = benign, 4.0/5.0 = pathogenic, 3.0 = VUS
        "LOF":          ["4.0", "5.0"],
        "Functional":   ["1.0", "2.0"],
        "Intermediate": ["3.0"],
    },
    "result_bouwman1_enigma_brca12_functional_assays": {
        "LOF":          ["Deleterious"],
        "Functional":   ["Neutral", "Likely not pathogenic"],
        "Intermediate": ["Not Clear"],
    },
    # ── BRCA2 assays ───────────────────────────────────────────────────
    "result_mesman_enigma_brca12_functional_assays": {
        "LOF":          ["Loss of function"],
        "Functional":   ["Functional"],
        "Intermediate": ["Intermediate function"],
    },
    "result_richardson_enigma_brca12_functional_assays": {
        "LOF":          ["Damaging"],
        "Functional":   ["Neutral"],
    },
    "result_biwas_enigma_brca12_functional_assays": {
        "LOF":          ["(NF)"],
        "Functional":   ["(F)"],
        "Intermediate": ["(I)"],
    },
    "result_ikegami_enigma_brca12_functional_assays": {
        "LOF":          ["fClass 4 or 5"],
        "Functional":   ["fClass 1", "fClass 1 or 2"],
        "Intermediate": ["fClass 3"],
    },
    # result_bouwman2: 只有 "Many Provided"，無法 binary mapping → 略過
}

def parse_functional_label(row):
    calls = []
    for col, mapping in assay_map.items():
        val = str(row[col]).strip()
        if val in ("-", "", "nan"):
            continue
        for label, keywords in mapping.items():
            if val in keywords:
                calls.append(label)
                break
    if not calls:
        return None
    counts = Counter(calls)
    if "LOF" in counts and "Functional" in counts:
        return "Conflicting"
    return counts.most_common(1)[0][0]

# ── 2. 篩選 missense substitution ────────────────────────────────────
is_missense = df["varconsequences"].str.contains("missense_variant", na=False)

df_missense = df[is_missense].copy()
print(f"Missense variants total: {len(df_missense)}")

# ── 3. 套用 functional label ─────────────────────────────────────────
df_missense["functional_label"] = df_missense.apply(parse_functional_label, axis=1)
print("\nFunctional label distribution:")
print(df_missense["functional_label"].value_counts(dropna=False))

# ── 4. Clinical significance ─────────────────────────────────────────
def classify_clin(val):
    if pd.isna(val) or str(val).strip() in ("-", ""): return None
    v = str(val).lower()
    if "likely pathogenic" in v: return "Likely_Pathogenic"
    if "pathogenic" in v:        return "Pathogenic"
    if "likely benign" in v:     return "Likely_Benign"
    if "benign" in v:            return "Benign"
    if "uncertain" in v or "vus" in v: return "VUS"
    return "Other"

# ENIGMA 優先，沒有才用 ClinVar
df_missense["clin_label"] = df_missense["clinical_significance_enigma"].apply(classify_clin)
no_enigma = df_missense["clin_label"].isna()
df_missense.loc[no_enigma, "clin_label"] = \
    df_missense.loc[no_enigma, "clinical_significance_clinvar"].apply(classify_clin)

print("\nClinical label distribution:")
print(df_missense["clin_label"].value_counts(dropna=False))

# ── 5. 產出 benchmark set ─────────────────────────────────────────────
benchmark = df_missense[
    df_missense["functional_label"].isin(["LOF", "Functional"]) &
    df_missense["clin_label"].isin(["Pathogenic", "Likely_Pathogenic", "Benign", "Likely_Benign"])
].copy()

print(f"\n=== Benchmark set: {len(benchmark)} variants ===")
print(benchmark[["clin_label", "functional_label"]].value_counts())

# ── 6. 輸出 ───────────────────────────────────────────────────────────
keep_cols = [
    "gene_symbol", "genomic_hgvs_38", "cdna", "protein",
    "varconsequences", "clin_label", "functional_label",
] + list(assay_map.keys())

# benchmark[keep_cols].reset_index(drop=True).to_csv(
#     "brca_functional_benchmark.csv", index=False
# )
# print("\nSaved: brca_functional_benchmark.csv")

In [ ]:
# 排除 discordant
concordant = benchmark[
    ~(
        (benchmark["clin_label"].isin(["Pathogenic", "Likely_Pathogenic"]) &
         (benchmark["functional_label"] == "Functional")) |
        ((benchmark["clin_label"] == "Benign") &
         (benchmark["functional_label"] == "LOF"))
    )
].copy()

# Binary label
concordant["label"] = concordant["clin_label"].map({
    "Pathogenic": 1,
    "Likely_Pathogenic": 1,
    "Benign": 0,
    "Likely_Benign": 0,
})

print("=== Final benchmark ===")
print(f"Total: {len(concordant)}")
print(f"Positive (LOF/P): {(concordant['label']==1).sum()}")
print(f"Negative (Func/B): {(concordant['label']==0).sum()}")

# 記錄每個 variant 來自哪個 assay（方便 reviewer 追溯）
def get_supporting_assays(row):
    assays = []
    for col, mapping in assay_map.items():
        val = str(row[col]).strip()
        if val in ("-", "", "nan"):
            continue
        for keywords in mapping.values():
            if val in keywords:
                assay_name = col.replace(
                    "_enigma_brca12_functional_assays", ""
                ).replace("result_", "")
                assays.append(assay_name)
                break
    return ";".join(assays)

concordant["supporting_assays"] = concordant.apply(get_supporting_assays, axis=1)
concordant["n_assays"] = concordant["supporting_assays"].str.count(";") + 1

# 輸出
keep_cols = [
    "gene_symbol", "genomic_hgvs_38", "cdna", "protein",
    "clin_label", "functional_label", "label",
    "supporting_assays", "n_assays",
] + list(assay_map.keys())

concordant[keep_cols].reset_index(drop=True).to_csv(
    "brca_functional_benchmark_final.csv", index=False
)

# 簡單 QC
print("\n=== Assay coverage ===")
print(concordant["n_assays"].value_counts().sort_index())
print("\n=== Gene distribution ===")
print(concordant["gene_symbol"].value_counts())

concordant.to_csv(os.path.join(root_path, "data/brca_functional_benchmark_final.csv"), index=False)

In [ ]:
concordant

In [ ]:
print(concordant["genomic_vcf38"].head(10).tolist())
print(concordant[["genomic_vcf38", "ref", "alt"]].head(10))

In [ ]:
dbNSFP

In [ ]:
import pandas as pd
import glob

# ── 1. 載入 chr13 + chr17 的 dbNSFP ─────────────────────────────────
dfs = []
for num in ["13", "17"]:
    df = pd.read_csv(os.path.join(f"/gpfs/home/pl2948/P_KNN/dbNSFP/hg38_missense_dbNSFP_chr{num}.csv"),
                     # comment="#", 
                     low_memory=False)
    df.columns = df.columns.str.strip()
    df["Chr"] = f"chr{num}"
    dfs.append(df)

dbNSFP = pd.concat(dfs, ignore_index=True)
dbNSFP["Start"] = dbNSFP["Start"].astype(int)

# ── 2. 從 concordant 解析座標 ────────────────────────────────────────
import re

def parse_vcf38(val):
    m = re.match(r'(chr\w+):g\.(\d+):([A-Z*-]+)>([A-Z*-]+)', str(val))
    if not m:
        return None, None, None, None
    return m.group(1), int(m.group(2)), m.group(3), m.group(4)

concordant[["Chr", "Start", "Ref", "Alt"]] = concordant["genomic_vcf38"].apply(
    lambda x: pd.Series(parse_vcf38(x))
)
concordant["Start"] = concordant["Start"].astype("Int64")

# ── 3. Merge ─────────────────────────────────────────────────────────
score_cols = [
    "BayesDel_noAF_score", "MutPred2_score", "REVEL_score",
    "VEST4_score", "AlphaMissense_score", "ESM1b_score", "VARITY_R_score",
]

merged = concordant.merge(
    dbNSFP[["Chr", "Start", "Ref", "Alt"] + score_cols],
    on=["Chr", "Start", "Ref", "Alt"],
    how="left"
)

print(f"Total: {len(merged)}")
print("\n=== Score coverage ===")
for col in score_cols:
    n = merged[col].notna().sum()
    print(f"{col}: {n}/{len(merged)} ({n/len(merged)*100:.1f}%)")

# ── 4. 直接存檔 ───────────────────────────────────────────────────────
merged.to_csv(os.path.join(root_path, "data", "brca_functional_benchmark_final.csv"), index=False)
print("\nSaved: brca_functional_benchmark_final.csv")